# Data Preprocessing

이 노트북은 원본 SDM-ART Excel 데이터를 XGBoost 학습에 사용할 수 있는 tabular dataset으로 변환합니다.

처리 흐름:
1. `data/raw/C210033_20250731_항목별.xlsx` 로드
2. 전원 대상자 ID 통합
3. 결측/이상값 정리
4. 투석 유형 label 생성
5. 인구통계, 지역, Visit 7 자가진단 설문 feature 생성
6. feature 병합, Visit 7 설문 작성 대상 유지, 결측값 처리
7. stratified train/test split
8. `data/processed/`에 전처리 결과 저장

Label 정의:
- `0`: 혈액투석
- `1`: 복막투석


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import re
from functools import reduce
from sklearn.model_selection import train_test_split


current_dir = Path.cwd().resolve()
for candidate in [current_dir, *current_dir.parents]:
    if (candidate / "data" / "raw").exists() and (candidate / "data_preprocessing").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("프로젝트 루트를 찾을 수 없습니다. 프로젝트 폴더 내부에서 노트북을 실행하세요.")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
RAW_EXCEL_PATH = RAW_DATA_DIR / "C210033_20250731_항목별.xlsx"

if not RAW_EXCEL_PATH.exists():
    raise FileNotFoundError(f"원본 Excel 파일을 찾을 수 없습니다: {RAW_EXCEL_PATH}")

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

sheets = pd.ExcelFile(RAW_EXCEL_PATH).sheet_names
df_dict = pd.read_excel(RAW_EXCEL_PATH, sheet_name=sheets)


In [ ]:
def load_transfer_cases() -> dict:
    """전원 대상자 ID 매핑(`data/raw/transfer_cases.json`)을 불러옵니다. 파일이 없으면 빈 매핑을 반환합니다."""
    transfer_path = RAW_DATA_DIR / "transfer_cases.json"
    if not transfer_path.exists():
        print(f"전원 대상자 매핑 파일이 없어 ID 통합을 건너뜁니다: {transfer_path}")
        return {}
    return json.loads(transfer_path.read_text(encoding="utf-8"))


TRANSFER_CASES = load_transfer_cases()


def transfer(df: pd.DataFrame) -> pd.DataFrame:
    """전원 대상자의 연구대상자ID를 최종 연구기관 기준 ID로 통합합니다."""
    df = df.copy()
    df["연구대상자ID"] = df["연구대상자ID"].replace(TRANSFER_CASES)
    return df


In [ ]:
df_dict = {s: transfer(d) for s, d in df_dict.items()}

In [ ]:
def clean_outliers_by_extremes(dataframe: pd.DataFrame) -> pd.DataFrame:
    """응답 거부/모름 값과 극단 sentinel 값을 결측값으로 변환합니다.

    숫자형 컬럼에서 sentinel 값이 해당 컬럼의 최솟값 또는 최댓값인 경우에만
    이상값으로 판단해 NaN으로 치환합니다.
    """
    df = dataframe.copy()
    df = df.replace(["모름", "무응답"], np.nan)

    outlier_values = [8888, 8888.0, 888, 888.0, 88.8, 888.88, 8888888888, 8888888888.0]

    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            min_val = df[col].min()
            max_val = df[col].max()
            df[col] = df[col].apply(
                lambda x: np.nan if x in outlier_values and (x == min_val or x == max_val) else x
            )

    return df


In [ ]:
df_dict = {sheet_name: clean_outliers_by_extremes(df) for sheet_name, df in df_dict.items()}

In [ ]:
def get_dialysis_treatment_label(df_dict: dict) -> pd.DataFrame:
    """투석 및 이식 sheet에서 혈액투석/복막투석 binary label을 생성합니다.

    - 0: 혈액투석
    - 1: 복막투석
    - 신장이식 케이스는 제외합니다.
    - 혈액투석과 복막투석이 모두 '예'인 경우 복막투석으로 간주합니다.
    """
    try:
        df = df_dict["22_투석 및 이식"].copy()

        dial_cols = ["혈액투석여부", "복막투석여부", "신장이식여부"]
        df[dial_cols] = df[dial_cols].fillna("아니오")

        df = df[df["신장이식여부"] != "예"].copy()
        keep_mask = (df["혈액투석여부"] == "예") | (df["복막투석여부"] == "예")
        df = df.loc[keep_mask].copy()
        df["labels"] = (df["복막투석여부"] == "예").astype(int)

        return df[["연구대상자ID", "labels"]]

    except KeyError as e:
        print(f"필수 sheet를 찾을 수 없습니다: {e}")
        return pd.DataFrame()


In [ ]:
df_label = get_dialysis_treatment_label(df_dict)
df_label

In [ ]:
def preprocess_age(df_dict: dict) -> pd.DataFrame:
    """인구통계 sheet에서 연령/성별 feature를 생성합니다."""
    try:
        df = df_dict["1_인구통계학적 정보"].copy()

        df["나이"] = pd.to_numeric(df["나이"], errors="coerce")
        df["age>=65"] = (df["나이"] >= 65).astype(int)
        df.loc[df["나이"].isna(), "age>=65"] = 0

        gender_map = {"Male": 1, "Female": 0}
        df["gender_male"] = df["성별"].map(gender_map).fillna(0).astype(int)

        return df[["연구대상자ID", "age>=65", "gender_male"]].copy()

    except KeyError as e:
        print(f"필수 sheet를 찾을 수 없습니다: {e}")
        return pd.DataFrame()


In [ ]:
df_age = preprocess_age(df_dict)

df_age

In [ ]:
def preprocess_region(df_dict: dict) -> pd.DataFrame:
    """
    17_설문지_비용조사

    Output:
        - 연구대상자ID
        - region_수도권: 1=수도권, 0=비수도권 (NaN 없음)
        - region_missing: 1=원본 지역이 결측/공백, 0=아님
        - region_norm: 정규화된 지역 문자열(체크용)
        - region_unknown_text: 1=비표준 텍스트/정규화 실패를 0으로 강제한 케이스, 0=아님
    """
    df = df_dict['17_설문지_비용조사'].copy()
    col = '거주지역_특별시/광역시/도'

    CANON = {
        "서울특별시", "경기도", "인천광역시",
        "강원도",
        "충청남도", "충청북도", "대전광역시", "세종특별자치시",
        "전라남도", "전라북도", "광주광역시",
        "경상남도", "경상북도", "부산광역시", "대구광역시", "울산광역시",
        "제주특별자치도",
    }
    CAPITAL = {"서울특별시", "경기도", "인천광역시"}

    EXACT_MAP = {
        # 수도권
        "서울": "서울특별시",
        "서울시": "서울특별시",
        "서울특별시": "서울특별시",
        "인천": "인천광역시",
        "인천시": "인천광역시",
        "인천광역시": "인천광역시",
        "경기": "경기도",
        "경기도": "경기도",
        "성남시": "경기도",
        "수원시": "경기도",
        "하남시": "경기도",

        # 강원/제주
        "강원": "강원도",
        "강원도": "강원도",
        "강원특별자치도": "강원도",
        "제주": "제주특별자치도",
        "제주도": "제주특별자치도",
        "제주특별자치도": "제주특별자치도",

        # 충청
        "충남": "충청남도",
        "충청남": "충청남도",
        "충청남도": "충청남도",
        "충북": "충청북도",
        "충청북": "충청북도",
        "충청북도": "충청북도",
        "충청": "충청북도",
        "대전": "대전광역시",
        "대전전": "대전광역시",
        "대전광역시": "대전광역시",
        "세종": "세종특별자치시",
        "세종시": "세종특별자치시",
        "세종특별자치시": "세종특별자치시",

        # 호남
        "광주": "광주광역시",
        "광주시": "광주광역시",
        "광주광역시": "광주광역시",
        "전남": "전라남도",
        "전라남": "전라남도",
        "전라남도": "전라남도",
        "전북": "전라북도",
        "전라북": "전라북도",
        "전라북도": "전라북도",

        # 영남
        "부산": "부산광역시",
        "부산시": "부산광역시",
        "부산광역시": "부산광역시",
        "대구": "대구광역시",
        "대구시": "대구광역시",
        "대구광역시": "대구광역시",
        "울산": "울산광역시",
        "울산시": "울산광역시",
        "울산광역시": "울산광역시",
        "경남": "경상남도",
        "경상남": "경상남도",
        "경상남도": "경상남도",
        "경남특별시": "경상남도",
        "경북": "경상북도",
        "경상북": "경상북도",
        "경상북도": "경상북도",

        # 비표준 응답 텍스트 처리
        "지방": "__FORCE_NON_CAPITAL__",
    }

    def _normalize_region(x: object) -> str:
        if pd.isna(x):
            return ""
        s = str(x).strip()
        if s.lower() in {"nan", "none"}:
            return ""
        s = re.sub(r"\s+", "", s)
        return EXACT_MAP.get(s, s)

    raw = df[col]
    norm = raw.apply(_normalize_region)

    # 원본 지역값이 비어 있던 케이스를 별도로 표시합니다.
    region_missing = (raw.isna() | raw.astype(str).str.strip().isin(["", "nan", "None", "none", "NaN"])).astype(int)

    # 정규화되지 않은 비표준 텍스트를 추적합니다.
    forced_non_cap = (norm == "__FORCE_NON_CAPITAL__")
    unknown_text = ((norm != "") & (~norm.isin(CANON)) & (~forced_non_cap)).astype(int)

    # 수도권 여부는 결측 없이 0/1로 유지합니다.
    region_capital = np.zeros(len(df), dtype=int)
    region_capital[norm.isin(CAPITAL)] = 1

    out = pd.DataFrame({
        "연구대상자ID": df["연구대상자ID"],
        "region_수도권": region_capital.astype(int),
        "region_missing": region_missing.astype(int),
        "region_norm": norm,
        "region_unknown_text": unknown_text.astype(int),
    })
    return out


In [ ]:
df_region = preprocess_region(df_dict)

unknown_mask = df_region["region_unknown_text"] == 1
unknown_counts = df_region.loc[unknown_mask, "region_norm"].value_counts()

print("정규화되지 않은 지역 응답:")
print(unknown_counts.head(50))

print("\nunknown_text 행 수:", int(unknown_mask.sum()))
print("unknown_text 고유값 수:", int(unknown_counts.shape[0]))
print("원본 지역 결측 행 수:", int(df_region["region_missing"].sum()))


In [ ]:
df_region_1row = (
    df_region
    .groupby("연구대상자ID", as_index=False)
    .agg(
        region_수도권=("region_수도권", "max"),
        region_missing=("region_missing", "max"),
        region_norm=("region_norm", lambda s: s.value_counts().index[0] if len(s.dropna()) > 0 else ""),
    )
)

# 대상자별 대표 지역값을 만든 뒤, 실제 지역명이 있으면 missing을 0으로 보정합니다.
df_region_1row.loc[:, "region_missing"] = (
    df_region_1row["region_norm"].astype(str).str.strip() == ""
).astype(int)

df_region_1row


In [ ]:
def keep_visit7_only(df: pd.DataFrame) -> pd.DataFrame:
    """Visit 7 / EoT 행만 남깁니다."""
    return df.loc[df["Visit명"].isin(["Visit 7 / EoT"])].copy()


In [ ]:
df_dict = {k: keep_visit7_only(v) for k, v in df_dict.items()}


In [ ]:
def preprocess_survey_diagnosis_18(df_dict: dict) -> pd.DataFrame:
    """Visit 7 자가진단도구 sheet에서 18문항 feature를 생성합니다."""
    try:
        df = df_dict['14_설문지_교육 후 자가진단도구'].copy()
        df = df.replace({'Yes': 1, 'No': 0}).infer_objects()

        # 연구대상자ID와 설문 응답 컬럼만 남깁니다.
        drop_cols = ['과제번호', '기관코드', '기관명', '연구대상자명', 'Random 번호', 'Random 배정일시',
                    'Random 배정자', 'Arm', 'Factors', '서면동의일', '일정시작일', '생년월일',
                    '나이단위', '연구대상자 등록일', 'Visit 반복번호', '방문예정일',
                    '실제방문일', 'eCRF명', 'eCRF Version', 'eCRF 반복번호', 'eCRF작성상태', '최초등록자',
                    '최초생성일', '최종수정자', '최종수정일', 'PI Confirm등록일', 'PI Confirm등록자',
                    'DM Confirm등록일', 'DM Confirm등록자', 'Lock등록일', 'Lock등록자','성별','나이','연구대상자상태', 'Visit명', 'SDM 교육진행여부', 'Intensive SDM 교육진행여부']

        df = df.drop(columns=drop_cols, errors='ignore')

        # 모델 입력에서 제외할 문항을 제거합니다.
        drop_cols_17 = ['7. 나는 매일 투석을 하고 싶지는 않다.',
        '8. 나는 정기적으로 의사의 진료를 받고 싶다.',
        '13. 나는 가능한 한 병원을 덜 가고 싶다.',
        '15. 나는 익숙한 것이 좋다.',
        '16. 나는 환경 변화를 싫어한다.',
        '17. 나는 새로운 것을 경험하는 것을 좋아한다.',
        '18. 나는 TV 보는 것을 좋아한다.',
        '19. 나는 오랜 시간 누워있는 것이 힘들다.',
        '22. 나는 내 일상을 배우자에게 많이 의존하는 편이다.',
        '25. 나는 가능한 많은 시간을 가족과 보내고 싶다.',
        '26. 나는 돌봐야 할 노부모와 함께 살고 있다.',
        '27. 나는 아이를 키우고 있다.',
        '29. 나는 내가 좋아하는 것을 먹고 마실 수 있어야 한다.',
        '30. 나는 땀을 많이 흘리기 때문에 목욕을 자주 하는 편이다.',
        '31. 나는 외모에 신경 쓰는 편이다.',
        '33. 나에게는 나를 돌봐 줄 사람이 없다.',
        '35. 나는 해외여행을 많이 다닌다.']

        df = df.drop(columns=drop_cols_17, errors='ignore')

        df['[자가진단도구] 설문지를 작성하였습니까?'] = df['[자가진단도구] 설문지를 작성하였습니까?'].fillna(0)
        
        reverse_cols = [
            '7. 나는 매일 투석을 하고 싶지는 않다.',
            '14. 나는 전문가의 투석이 더 편하다.',
            '16. 나는 환경 변화를 싫어한다.',
            '20. 나는 주삿바늘을 무서워한다.',
            '21. 나는 남의 눈을 많이 의식하는 편이다.',
            '22. 나는 내 일상을 배우자에게 많이 의존하는 편이다.',
            '23. 나는 나를 보살펴 줄 사람이 필요하다.',
            '24. 나는 투석기계를 내 집에 들어놓고 싶지 않다.',
            '32. 나는 새로운 것에 익숙해지기까지 시간이 많이 걸린다.',
            '33. 나에게는 나를 돌봐 줄 사람이 없다.'
        ]

        # 낮은 점수가 높은 선호도를 의미하는 문항은 reverse coding합니다.
        for col in reverse_cols:
            if col in df.columns:
                df[col] = 6 - df[col]
                new_col_name = f'{col}_rev'
                df.rename(columns={col: new_col_name}, inplace=True)

        # 나머지 설문 결측치는 Likert 중간값인 3으로 대체합니다.
        df = df.fillna(3)


        return df

    except KeyError as e:
        print(f"필수 sheet를 찾을 수 없습니다: {e}")
        return pd.DataFrame()


In [ ]:
df_survey_diagnosis = preprocess_survey_diagnosis_18(df_dict)
df_survey_diagnosis

In [ ]:
def merge_all(dfs):
    """연구대상자ID 기준으로 feature table들을 outer merge합니다."""
    return reduce(
        lambda left, right: pd.merge(
            left,
            right.drop(columns=[c for c in right.columns if c != "연구대상자ID" and c in left.columns]),
            on="연구대상자ID",
            how="outer",
        ),
        dfs,
    )


In [ ]:
merge_target = [df_label, df_age, df_region_1row, df_survey_diagnosis]
df_merge_all = merge_all(merge_target)
df_merge_all = df_merge_all.applymap(lambda x: 1 if x is True else (0 if x is False else x))

df_merge_all.dropna(subset=['labels'], inplace=True)

df_merge_all.columns = [col.replace(',', '') for col in df_merge_all.columns]

df_merge_all['[자가진단도구] 설문지를 작성하였습니까?'] = df_merge_all['[자가진단도구] 설문지를 작성하였습니까?'].fillna(0)
df_merge_all = df_merge_all.loc[df_merge_all['[자가진단도구] 설문지를 작성하였습니까?'] == 1].copy()
df_merge_all['region_수도권'] = df_merge_all['region_수도권'].fillna(0).astype(int)
df_merge_all['region_missing'] = df_merge_all['region_missing'].fillna(1).astype(int)
df_merge_all['region_norm'] = df_merge_all['region_norm'].fillna('미상').astype(str)
df_merge_all = df_merge_all.fillna(3)

df_merge_all

In [ ]:
all_csv_path = PROCESSED_DATA_DIR / "df_merge_all_diagnosis_18.csv"
all_xlsx_path = PROCESSED_DATA_DIR / "df_merge_all_diagnosis_18.xlsx"

df_merge_all.to_csv(all_csv_path, index=False, encoding="utf-8")
df_merge_all.to_excel(all_xlsx_path, index=False)


# Train/Test Split

Label, age group, region bucket 분포를 최대한 유지하도록 stratified split을 수행합니다.


In [ ]:
df_temp = df_merge_all.copy()

# Stratification에 사용할 컬럼을 숫자형/범주형으로 정리합니다.
df_temp["labels"] = pd.to_numeric(df_temp["labels"], errors="coerce")
df_temp["age>=65"] = pd.to_numeric(df_temp["age>=65"], errors="coerce").fillna(0).astype(int)

if "region_수도권" not in df_temp.columns:
    raise ValueError("region_수도권 column is missing in df_merge_all.")
if "region_missing" not in df_temp.columns:
    raise ValueError("region_missing column is missing in df_merge_all.")

df_temp["region_missing"] = pd.to_numeric(df_temp["region_missing"], errors="coerce").fillna(0).astype(int)

# 지역은 수도권/비수도권/미상으로 나누어 split 분포를 맞춥니다.
df_temp["region_bucket"] = np.where(
    df_temp["region_missing"] == 1,
    "missing",
    np.where(pd.to_numeric(df_temp["region_수도권"], errors="coerce") == 1, "capital", "non_capital"),
)

stratify_keys = ["labels", "age>=65", "region_bucket"]
df_temp["stratify_key"] = df_temp[stratify_keys].astype(str).agg("_".join, axis=1)

# stratified split이 불가능한 희소 조합은 제외합니다.
key_counts = df_temp["stratify_key"].value_counts()
keys_to_remove = key_counts[key_counts < 2].index

df_splittable = df_merge_all.loc[~df_temp["stratify_key"].isin(keys_to_remove)].copy()
df_temp_splittable = df_temp.loc[~df_temp["stratify_key"].isin(keys_to_remove)].copy()

train_df, test_df = train_test_split(
    df_splittable,
    test_size=0.1,
    random_state=42,
    stratify=df_temp_splittable["stratify_key"],
)

print("Stratified split done")
print(f"Train set size: {len(train_df)} rows")
print(f"Test set size:  {len(test_df)} rows")

print("\n--- labels distribution ---")
print("Original:\n", df_merge_all.loc[df_splittable.index, "labels"].value_counts(normalize=True).sort_index())
print("\nTrain Set:\n", train_df["labels"].value_counts(normalize=True).sort_index())
print("\nTest Set:\n", test_df["labels"].value_counts(normalize=True).sort_index())

print("\n--- age>=65 distribution ---")
print("Original:\n", df_merge_all.loc[df_splittable.index, "age>=65"].value_counts(normalize=True).sort_index())
print("\nTrain Set:\n", train_df["age>=65"].value_counts(normalize=True).sort_index())
print("\nTest Set:\n", test_df["age>=65"].value_counts(normalize=True).sort_index())

print("\n--- region_bucket distribution ---")

def _region_bucket(df: pd.DataFrame) -> np.ndarray:
    rm = pd.to_numeric(df["region_missing"], errors="coerce").fillna(0).astype(int)
    rc = pd.to_numeric(df["region_수도권"], errors="coerce")
    return np.where(rm == 1, "missing", np.where(rc == 1, "capital", "non_capital"))

print("Original:\n", pd.Series(_region_bucket(df_merge_all.loc[df_splittable.index])).value_counts(normalize=True))
print("\nTrain Set:\n", pd.Series(_region_bucket(train_df)).value_counts(normalize=True))
print("\nTest Set:\n", pd.Series(_region_bucket(test_df)).value_counts(normalize=True))

train_csv_path = PROCESSED_DATA_DIR / "df_merge_train_diagnosis_18.csv"
test_csv_path = PROCESSED_DATA_DIR / "df_merge_test_diagnosis_18.csv"
train_xlsx_path = PROCESSED_DATA_DIR / "df_merge_train_diagnosis_18.xlsx"
test_xlsx_path = PROCESSED_DATA_DIR / "df_merge_test_diagnosis_18.xlsx"

train_df.to_csv(train_csv_path, index=False, encoding="utf-8")
test_df.to_csv(test_csv_path, index=False, encoding="utf-8")
train_df.to_excel(train_xlsx_path, index=False)
test_df.to_excel(test_xlsx_path, index=False)


In [ ]:
# df_merge_*_diagnosis_18.csv의 컬럼명을 영어로 바꾼 사본을 저장합니다.
column_name_map = {
    "연구대상자ID": "ParticipantID",
    "labels": "labels",
    "age>=65": "age>=65",
    "gender_male": "gender_male",
    "region_수도권": "region_capital",
    "region_missing": "region_missing",
    "region_norm": "region_norm",
    "[자가진단도구] 설문지를 작성하였습니까?": "[Self-diagnosis tool] Did you complete the questionnaire?",
    "1. 나는 내 심장을 보호하고 싶다.": "1. I want to protect my heart.",
    "2. 나는 나의 뼈 관절과 신경을 보호하고 싶다.": "2. I want to protect my bones joints and nerves.",
    "3. 나는 나의 의지대로 살고 싶다.": "3. I want to live according to my will.",
    "4. 나는 내 가족에게 짐이 되고 싶지 않다.": "4. I do not want to be a burden to my family.",
    "5. 나는 가능한 한 높은 삶의 질을 유지하고 싶다.": "5. I want to maintain the highest possible quality of life.",
    "6. 나는 하루 정도는 투석 부담 없이 지내고 싶다.": "6. I want to live for about a day without the burden of dialysis.",
    "9. 나는 내 스스로가 내 일상을 조절할 수 있다.": "9. I can control my daily life by myself.",
    "10. 나는 규칙적인 생활을 한다.": "10. I live a regular life.",
    "11. 나는 일상에 계획을 세우고 행동한다.": "11. I make plans for my daily life and act on them.",
    "12. 나는 운동을 즐긴다.": "12. I enjoy exercising.",
    "14. 나는 전문가의 투석이 더 편하다._rev": "14. Dialysis by a specialist is more comfortable._rev",
    "20. 나는 주삿바늘을 무서워한다._rev": "20. I am afraid of needles._rev",
    "21. 나는 남의 눈을 많이 의식하는 편이다._rev": "21. I am very conscious of what others think._rev",
    "23. 나는 나를 보살펴 줄 사람이 필요하다._rev": "23. I need someone to take care of me._rev",
    "24. 나는 투석기계를 내 집에 들어놓고 싶지 않다._rev": "24. I do not want to have a dialysis machine in my house._rev",
    "28. 나는 학교에 가거나 직장에 다녀야 한다.": "28. I have to go to school or work.",
    "32. 나는 새로운 것에 익숙해지기까지 시간이 많이 걸린다._rev": "32. It takes me a long time to get used to new things._rev",
    "34. 나는 여행을 좋아해서 절대 포기할 수 없다.": "34. I love traveling and can never give it up.",
}

english_export_targets = {
    "all": (
        PROCESSED_DATA_DIR / "df_merge_all_diagnosis_18.csv",
        PROCESSED_DATA_DIR / "df_merge_all_diagnosis_18_eng.csv",
    ),
    "train": (
        PROCESSED_DATA_DIR / "df_merge_train_diagnosis_18.csv",
        PROCESSED_DATA_DIR / "df_merge_train_diagnosis_18_eng.csv",
    ),
    "test": (
        PROCESSED_DATA_DIR / "df_merge_test_diagnosis_18.csv",
        PROCESSED_DATA_DIR / "df_merge_test_diagnosis_18_eng.csv",
    ),
}

saved_eng_paths = []
for split_name, (source_path, target_path) in english_export_targets.items():
    if not source_path.exists():
        raise FileNotFoundError(f"Source CSV for {split_name} split was not found: {source_path}")

    df_eng = pd.read_csv(source_path)

    missing_columns = [col for col in df_eng.columns if col not in column_name_map]
    if missing_columns:
        raise ValueError(f"{split_name} English column-name mapping is missing: {missing_columns}")

    df_eng = df_eng.rename(columns=column_name_map)
    df_eng.to_csv(target_path, index=False, encoding="utf-8")
    saved_eng_paths.append(target_path)

saved_eng_paths
